In [1]:
import pandas as pd

# Show every column when we print a table (pandas hides some by default)
pd.set_option("display.max_columns", None)

# The notebook lives in notebooks/, so ".." means "go up one folder"
DATA_PATH = "../data/raw/charging_record.csv"
df = pd.read_csv(DATA_PATH)

print("Rows, columns:", df.shape)

Rows, columns: (72856, 13)


In [2]:
df.head()

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54,2022-09-15 23:59,185,20.36
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01,2022-09-14 21:31,90,10.19
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54,2022-09-14 19:54,60,6.78
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32,2022-09-30 0:16,344,37.65
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30,2022-09-26 0:30,300,33.81


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 72856 entries, 0 to 72855
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   UserID          72856 non-null  int64  
 1   ChargerID       72856 non-null  int64  
 2   ChargerCompany  72856 non-null  int64  
 3   Location        72856 non-null  str    
 4   ChargerType     72856 non-null  int64  
 5   StartDay        72856 non-null  str    
 6   StartTime       72856 non-null  str    
 7   EndDay          72856 non-null  str    
 8   EndTime         72856 non-null  str    
 9   StartDatetime   72856 non-null  str    
 10  EndDatetime     72856 non-null  str    
 11  Duration        72856 non-null  int64  
 12  Demand          72856 non-null  float64
dtypes: float64(1), int64(5), str(7)
memory usage: 12.5 MB


In [4]:
df.describe()

,UserID,ChargerID,ChargerCompany,ChargerType,Duration,Demand
count,72856.000000,72856.000000,72856.000000,72856.00000,72856.000000,72856.000000
mean,338.164667,525.489212,0.627443,0.20524,151.194054,17.444445
std,553.970996,569.544469,0.483489,0.40388,146.430286,13.492728
min,0.000000,1.000000,0.000000,0.00000,-29.000000,0.010000
25%,0.000000,107.000000,0.000000,0.00000,40.000000,7.530000
50%,62.000000,301.000000,1.000000,0.00000,112.000000,14.100000
75%,407.000000,738.000000,1.000000,0.00000,202.000000,23.200000
max,2501.000000,2671.000000,1.000000,1.00000,1573.000000,97.000000


In [5]:
missing = df.isna().sum()
print("Columns with missing values:")
print(missing[missing > 0] if missing.any() else "None")

print("\nDuplicate rows:", df.duplicated().sum())

Columns with missing values:
None

Duplicate rows: 13


In [6]:
df.nunique()

UserID             2337
ChargerID          2119
ChargerCompany        2
Location             14
ChargerType           2
StartDay            366
StartTime         43245
EndDay              365
EndTime           46710
StartDatetime     65628
EndDatetime       66745
Duration            979
Demand             6046
dtype: int64

In [7]:
start = pd.to_datetime(df["StartDatetime"], errors="coerce")
end = pd.to_datetime(df["EndDatetime"], errors="coerce")

print("Date range:", start.min(), "to", start.max())
print("Start dates that could not be read:", start.isna().sum())
print("Sessions that end before they start:", (end < start).sum())

Date range: 2021-09-30 17:14:00 to 2022-09-30 23:11:00
Start dates that could not be read: 0
Sessions that end before they start: 17


In [8]:
calculated_minutes = (end - start).dt.total_seconds() / 60

check = pd.DataFrame({
    "Duration": df["Duration"],
    "calculated_minutes": calculated_minutes,
})
check.head(10)

,Duration,calculated_minutes
0,185,185.0
1,90,90.0
2,60,60.0
3,344,344.0
4,300,300.0
5,70,70.0
6,60,60.0
7,494,494.0
8,60,60.0
9,112,112.0


In [9]:
df[["Demand", "Duration"]].corr()

,Demand,Duration
Demand,1.00000,0.65641
Duration,0.65641,1.00000


In [10]:
median_demand = df["Demand"].median()
is_high = df["Demand"] >= median_demand

print("Median demand:", median_demand)
print(is_high.value_counts(normalize=True).round(3))

Median demand: 14.1
Demand
True     0.5
False    0.5
Name: proportion, dtype: float64
